# AeroVision — Train pix2pix-turbo (clean airfoil → airflow)

Run this on a **GPU runtime** (Colab: Runtime → Change runtime type → GPU).
It trains the model on the 193 paired images and produces a checkpoint
`.pkl` file that the backend loads via the `AEROVISION_MODEL_PATH` env var.

Steps: install → train → download the `.pkl`.

In [ ]:
# 1. Check GPU
!nvidia-smi -L

In [ ]:
# 2. Get the code + data
!git clone https://github.com/jamesvo2103/AeroVision.git
%cd AeroVision/ml_model

In [ ]:
# 3. Install dependencies (takes a few minutes)
!pip install -q -r requirements.txt
!pip install -q accelerate

In [ ]:
# 4. Train. Checkpoints are saved every 500 steps to
#    outputs/airfoil_airflow/checkpoints/model_<step>.pkl
#
# WANDB_MODE=offline avoids needing a Weights & Biases login.
# Lower --max_train_steps for a quick test; raise it for better quality.
import os
os.environ['WANDB_MODE'] = 'offline'

!accelerate launch --mixed_precision=fp16 src/train_pix2pix_turbo.py \
  --pretrained_model_name_or_path "stabilityai/sd-turbo" \
  --dataset_folder "data/Aerovision_data" \
  --output_dir "outputs/airfoil_airflow" \
  --resolution 512 \
  --train_batch_size 4 \
  --max_train_steps 3000 \
  --checkpointing_steps 500 \
  --eval_freq 500 \
  --viz_freq 500 \
  --learning_rate 5e-6 \
  --lambda_lpips 5 --lambda_l2 1 --lambda_gan 0.5 --lambda_clipsim 5 \
  --report_to "wandb" --tracker_project_name "aerovision" \
  --enable_xformers_memory_efficient_attention \
  --gradient_checkpointing

In [ ]:
# 5. Pick the latest checkpoint and download it
import glob, os
ckpts = sorted(glob.glob('outputs/airfoil_airflow/checkpoints/model_*.pkl'),
               key=os.path.getmtime)
print('Checkpoints:', ckpts)
latest = ckpts[-1]
print('Latest:', latest)

# In Colab this downloads the file to your computer:
try:
    from google.colab import files
    files.download(latest)
except Exception as e:
    print('Not on Colab — copy this file manually:', latest)

## After training
1. Put the downloaded `.pkl` somewhere on the serving machine, e.g.
   `ml_model/checkpoints/airfoil_airflow.pkl`.
2. Start the backend with the checkpoint path set:
   ```bash
   export AEROVISION_MODEL_PATH=ml_model/checkpoints/airfoil_airflow.pkl
   export AEROVISION_DEVICE=cuda   # needs a GPU
   uvicorn backend.main:app --port 8000
   ```
3. The backend switches from STUB mode to the real model automatically.